# Audio Processing Using Mel-Spectrograms 

A Mel-Spectrogram is a 2D heatmap that visualizes audio by showing **Time** vs **Frequency (Pitch)** vs **Volume (Color)**. 

Unlike standard waveforms, it displays audio exactly the way the human brain physically perceives and processes it.

## Core Concepts 

* **Amplitude & Intensity:** Amplitude is the physical height of the waveform. The higher the wave, the more intense the energy. Our brains perceive this physical intensity as *loudness*.

* **Logarithmic Human Hearing:** Human hearing is not linear; it is logarithmic. 
  * A difference of **100 Hz** at the low end example, 100 Hz -> 200 Hz is highly audible to the human ear.
  * However, that exact same mathematical difference at the high end (e.g., 10,000 Hz -> 10,100 Hz) sounds completely identical to us. The Mel Scale accounts for this biological quirk.

In [1]:
# Imports
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

# Use 'MS Gothic' for Windows, 'AppleGothic' for Mac, or 'Noto Sans CJK JP' for Linux
plt.rcParams['font.family'] = 'MS Gothic' 

# Quick Test
plt.title("ひらがなのテスト (Hiragana Test)")
plt.show()

ModuleNotFoundError: No module named 'librosa'

# Data and folder Setup

In [ ]:
# Environment Setup
audio_dir = "../ojad_audio/"
output_dir = "../mel_spectrograms/"

# exist_ok=True ensures a crash doesnt occur
os.makedirs(output_dir, exist_ok=True) 

# Search the directory and grab ONLY files ending in .mp3
audio_files = [f for f in os.listdir(audio_dir) if f.endswith('.mp3')]

print(f"Data Check: Found {len(audio_files)} MP3 files ready for processing.")
# check first file
if audio_files:
    print(f"Sample target: {audio_files[0]}")

# Transformation Logic

Sample Rate: sr = 22050hz is enough for standard human speech

Mel - Spectogram - the Audio is chopped into tiny slices of time called windows 
    And Analyzes the volume of 128 different frequency bands (y-axis)

Decibels - Neural networks learn better when data mimics human perception

Human hearing is logarithmic, 

In [ ]:
# Audio Processor Function
def mp3_to_mel_spectrogram(file_path):
    """
    Translates an MP3 file into a 2D mathematical Mel-Spectrogram matrix.
    Returns: A NumPy array representing the audio.
    """
    # Load the audio. 
    # audio_signal is a 1D array of the actual sound waves.
    audio_signal, sample_rate = librosa.load(file_path, sr=22050)
    
    # Perform the Fourier Transform to get frequencies
    mel_spec = librosa.feature.melspectrogram(
        y=audio_signal, 
        sr=sample_rate, 
        n_mels=128,  # We want 128 rows of frequency data (the Y-axis)
        fmax=8000    # Human speech pitch rarely goes above 8000Hz, cropped the high end static
    )
    
    # Convert the raw energy into a Logarithmic scale (Decibels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    return mel_spec_db, sample_rate

print("Audio processing ready")

In [ ]:
# Visual Sanity Check
if len(audio_files) > 0:
    test_file = audio_files[0]
    test_path = os.path.join(audio_dir, test_file)
    
    # Run our function on the single test file
    test_matrix, test_sr = mp3_to_mel_spectrogram(test_path)
    
    print(f"--- VISUAL CHECK: {test_file} ---")
    print(f"Mathematical Matrix Shape: {test_matrix.shape}")
    print(f"This means there are 128 frequency bands, and the audio was chopped into {test_matrix.shape[1]} slices of time.")
    
    # Draw the Heatmap
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(test_matrix, x_axis='time', y_axis='mel', sr=test_sr, fmax=8000)
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Mel-Spectrogram Heatmap: {test_file}')
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import numpy as np

# 1. THE BULLETPROOF PATHS (Stepping up one folder)
notebook_path = os.path.abspath('') 

# Tell Python to go UP one level to the main PitchAccentTrainerJP folder
parent_path = os.path.dirname(notebook_path)

# Now grab the audio from that main folder
audio_dir = os.path.join(parent_path, "ojad_audio")


# Safely grab all MP3s
audio_files = [f for f in os.listdir(audio_dir) if f.lower().endswith('.mp3')]

# 2. THE BATCH OVEN
success_count = 0
error_count = 0

print(f"Starting batch conversion for {len(audio_files)} files...")

for i, filename in enumerate(audio_files):
    file_path = os.path.join(audio_dir, filename)
    
    try:
        # Run the math function (Make sure your mp3_to_mel_spectrogram function cell is run first!)
        mel_matrix, _ = mp3_to_mel_spectrogram(file_path)
        
        # Strip the .mp3 and replace it with .npy 
        base_name = filename.replace('.mp3', '').replace('.MP3', '')
        save_path = os.path.join(output_dir, f"{base_name}.npy")
        
        # Save the raw matrix to the hard drive
        np.save(save_path, mel_matrix)
        success_count += 1
        
    except Exception as e:
        print(f"  -> Failed to process {filename}: {e}")
        error_count += 1

    # Print a progress update every 50 files
    if (i + 1) % 50 == 0:
        print(f"  ... Successfully converted {i + 1}/{len(audio_files)} files ...")

print("\n========================================")
print(f"PHASE 2 COMPLETE: {success_count} files converted successfully. ({error_count} errors)")
print(f"Your files are safely stored at: {output_dir}")
print("========================================")